In [ ]:
# P12: Fast GAN for MNIST Digit Generation
# Code by Parthiv Abhani

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

# ==========================================
# 1. Load MNIST
# ==========================================
(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()

# Use only 10,000 images for faster training
x_train = x_train[:10000]

# Normalize to [-1, 1]
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

# Smaller batch
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(10000).batch(BATCH_SIZE)


# ==========================================
# 2. Generator
# ==========================================
generator = tf.keras.Sequential([
    layers.Input(shape=(100,)),
    layers.Dense(7 * 7 * 64),
    layers.LeakyReLU(),

    layers.Reshape((7, 7, 64)),

    layers.Conv2DTranspose(
        32, 4, strides=2, padding="same"
    ),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(
        1, 4, strides=2,
        padding="same",
        activation="tanh"
    )
])


# ==========================================
# 3. Discriminator
# ==========================================
discriminator = tf.keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(
        32, 4, strides=2, padding="same"
    ),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(
        64, 4, strides=2, padding="same"
    ),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(1)
])


# ==========================================
# 4. Loss and Optimizers
# ==========================================
loss_fn = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

g_optimizer = tf.keras.optimizers.Adam(0.0002)
d_optimizer = tf.keras.optimizers.Adam(0.0002)


# ==========================================
# 5. Training Step
# ==========================================
@tf.function
def train_step(real_images):

    noise = tf.random.normal(
        [BATCH_SIZE, 100]
    )

    with tf.GradientTape() as gt, tf.GradientTape() as dt:

        fake_images = generator(
            noise,
            training=True
        )

        real_output = discriminator(
            real_images,
            training=True
        )

        fake_output = discriminator(
            fake_images,
            training=True
        )

        g_loss = loss_fn(
            tf.ones_like(fake_output),
            fake_output
        )

        d_loss = (
            loss_fn(
                tf.ones_like(real_output),
                real_output
            )
            +
            loss_fn(
                tf.zeros_like(fake_output),
                fake_output
            )
        )

    g_gradients = gt.gradient(
        g_loss,
        generator.trainable_variables
    )

    d_gradients = dt.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    g_optimizer.apply_gradients(
        zip(
            g_gradients,
            generator.trainable_variables
        )
    )

    d_optimizer.apply_gradients(
        zip(
            d_gradients,
            discriminator.trainable_variables
        )
    )

    return g_loss, d_loss


# ==========================================
# 6. Train GAN
# ==========================================
EPOCHS = 5

for epoch in range(EPOCHS):

    for images in dataset:

        g_loss, d_loss = train_step(images)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Generator Loss: {g_loss:.4f} | "
        f"Discriminator Loss: {d_loss:.4f}"
    )


# ==========================================
# 7. Generate Synthetic Images
# ==========================================
noise = tf.random.normal([16, 100])

generated_images = generator(
    noise,
    training=False
)

# Convert [-1, 1] to [0, 1]
generated_images = (generated_images + 1) / 2


# ==========================================
# 8. Display Generated Digits
# ==========================================
plt.figure(figsize=(8, 8))

for i in range(16):

    plt.subplot(4, 4, i + 1)

    plt.imshow(
        generated_images[i, :, :, 0],
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle("Generated Handwritten Digits")
plt.tight_layout()
plt.show()